In [ ]:
import pickle
import h5py
import os
import numpy as np
import json

In [ ]:
path = "../FMM/test_/Dict/test_None.pkl"

# Pickle

In [ ]:
size_bytes_pickle = os.path.getsize(path)
print("Pickle file size :", size_bytes_pickle)

# H5PY

In [ ]:
data = pickle.load(open(path, "rb"))

In [ ]:
parameters = data["parameters"]
# data["fsky"] = np.array(data["fsky"])
duration = data["duration"]
qubic_dict = data["qubic_dict"]

In [ ]:
with h5py.File("data.h5", "w") as h5f:
    grp_params = h5f.create_group("parameters")
    grp_qubic_dict = h5f.create_group("qubic_dict")
    for key, value in data.items():
        print(key)
        print(type(value))
        if key == "parameters":
            grp_params.attrs["parameters"] = json.dumps(value)
        elif key == "qubic_dict":
            grp_qubic_dict.attrs["qubic_dict"] = json.dumps(value)
        elif key in ["fsky", "duration"]:
            h5f.attrs[key] = value
        else:
            h5f.create_dataset(key, data=value, compression="gzip")

In [ ]:
size_bytes_h5 = os.path.getsize("data.h5")
print("h5py file size :", size_bytes_h5)

# Comparison

In [ ]:
print("Pickle file size :", size_bytes_pickle, " bytes.")
print("h5py file size :", size_bytes_h5, " bytes.")
print("Difference : ", size_bytes_pickle - size_bytes_h5, " bytes.")
print("h5 saves ", np.round((1 - size_bytes_h5 / size_bytes_pickle) * 100, 2), "% of memory")

# Test Qhdf5

In [ ]:
from qubic.lib.Qhdf5 import HDF5Dict

hdf5 = HDF5Dict()

In [ ]:
hdf5.save_dict("test.h5", data)

In [ ]:
dict_test = hdf5.load_dict("test.h5")

In [ ]:
print(data.keys())
print(dict_test["fsky"])
print(dict_test["nus"])
print(dict_test["qubic_dict"])
print(dict_test["qubic_dict"].keys())
print(dict_test["parameters"])

In [ ]:
print(hdf5.load_item("test.h5", "parameters/CMB"))

# Test reliability between HDF5 and Pickle

In [ ]:
planck143_pkl = pickle.load(open("../../../../../qubic/data/Planck143GHz.pkl", "rb"))["noise143"]

hdf5.save_array("planck_test.h5", planck143_pkl)
planck143_hdf5 = hdf5.load_array("planck_test.h5")

In [ ]:
print(np.array_equal(planck143_pkl, planck143_hdf5))
print(np.mean(planck143_pkl - planck143_hdf5))